# Video Clone — Colab remote worker

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daokimluc/video-clone-colab/blob/main/video_clone_colab_backend.ipynb)

Repo: https://github.com/daokimluc/video-clone-colab

## Cách dùng (desktop Video Clone)
1. **Runtime → Run all** (hoặc chạy lần lượt các cell bên dưới)
2. Cell **Tunnel** sẽ in ra `REMOTE_URL` dạng `https://….trycloudflare.com`
3. Copy **REMOTE_URL** + **SHARED_SECRET** vào app → **Cấu hình → Remote**
4. App probe: `GET {REMOTE_URL}/health` header `X-VC-Secret: {SHARED_SECRET}`

Desktop vẫn là control plane (projects/jobs). Notebook này chỉ chạy worker + tunnel.

## 1) Cài dependency + cloudflared

In [ ]:
# Python packages for worker
!pip -q install fastapi uvicorn httpx

# cloudflared binary (Linux Colab)
import os, subprocess, shutil

def ensure_cloudflared():
    if shutil.which('cloudflared'):
        print('cloudflared already on PATH:', shutil.which('cloudflared'))
        return
    print('Installing cloudflared…')
    # Official linux amd64 binary (Colab VMs are usually amd64)
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    dest = '/usr/local/bin/cloudflared'
    !wget -q -O /tmp/cloudflared {url}
    !chmod +x /tmp/cloudflared
    !cp /tmp/cloudflared {dest}
    print('cloudflared installed:', shutil.which('cloudflared') or dest)

ensure_cloudflared()
!cloudflared --version

## 2) Shared secret (dán vào app)

In [ ]:
import secrets

SHARED_SECRET = secrets.token_urlsafe(16)
print('=' * 60)
print('SHARED_SECRET (copy vào app → Shared secret):')
print(SHARED_SECRET)
print('=' * 60)

## 3) Start FastAPI worker (port 8765)

In [ ]:
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
import uvicorn
import threading
import time
import httpx

app = FastAPI(title='VideoClone Colab Worker')
WORKER_PORT = 8765

def check(secret: str | None):
    if secret != SHARED_SECRET:
        raise HTTPException(401, 'bad secret')

@app.get('/health')
def health(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'ok': True, 'worker': 'colab', 'port': WORKER_PORT}

@app.get('/capabilities')
def caps(x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {'asr': True, 'tts': True, 'worker': 'colab'}

class AsrReq(BaseModel):
    note: str = 'stub — wire faster-whisper here'

@app.post('/infer/asr')
def infer_asr(body: AsrReq, x_vc_secret: str | None = Header(default=None)):
    check(x_vc_secret)
    return {
        'status': 'not_implemented',
        'hint': 'Desktop still runs ASR locally until infer is wired',
        'note': body.note,
    }

def _run_uvicorn():
    uvicorn.run(app, host='127.0.0.1', port=WORKER_PORT, log_level='warning')

# avoid double-start if re-run cell
if not globals().get('_VC_WORKER_STARTED'):
    threading.Thread(target=_run_uvicorn, daemon=True).start()
    _VC_WORKER_STARTED = True
    time.sleep(1.2)

# local self-check
try:
    r = httpx.get(
        f'http://127.0.0.1:{WORKER_PORT}/health',
        headers={'X-VC-Secret': SHARED_SECRET},
        timeout=5.0,
    )
    print('Local worker health:', r.status_code, r.json())
except Exception as e:
    print('Local worker not ready yet:', e)

print(f'Worker listening on http://127.0.0.1:{WORKER_PORT}')

## 4) Cloudflare quick tunnel → **REMOTE_URL** (copy vào app)

Cell này tự chạy `cloudflared tunnel --url http://127.0.0.1:8765` và **bắt URL** `https://….trycloudflare.com`.

In [ ]:
import re
import subprocess
import time
import shutil
import httpx

assert shutil.which('cloudflared'), 'cloudflared missing — re-run install cell'
assert 'SHARED_SECRET' in globals(), 'Run SHARED_SECRET cell first'

# stop previous tunnel from this notebook if re-run
if globals().get('_VC_TUNNEL_PROC') is not None:
    try:
        _VC_TUNNEL_PROC.terminate()
    except Exception:
        pass

cmd = ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{WORKER_PORT}']
print('Starting:', ' '.join(cmd))
_VC_TUNNEL_PROC = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

REMOTE_URL = None
pattern = re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com')
deadline = time.time() + 90
lines = []

while time.time() < deadline and REMOTE_URL is None:
    line = _VC_TUNNEL_PROC.stdout.readline()
    if not line:
        if _VC_TUNNEL_PROC.poll() is not None:
            break
        time.sleep(0.1)
        continue
    line = line.rstrip()
    lines.append(line)
    # cloudflared prints the public URL in logs
    m = pattern.search(line)
    if m:
        REMOTE_URL = m.group(0).rstrip('/')
        break

if not REMOTE_URL:
    print('--- last cloudflared lines ---')
    print('\n'.join(lines[-30:]))
    raise RuntimeError(
        'Không bắt được URL trycloudflare.com. '
        'Chạy lại cell tunnel, hoặc kiểm tra mạng Colab / cloudflared.'
    )

print('=' * 60)
print('REMOTE_URL (copy vào app → Remote URL):')
print(REMOTE_URL)
print('SHARED_SECRET (copy vào app → Shared secret):')
print(SHARED_SECRET)
print('=' * 60)
print('App settings: Backend mode = Remote')
print()

# public health check through tunnel
try:
    time.sleep(1.5)
    hr = httpx.get(
        REMOTE_URL + '/health',
        headers={'X-VC-Secret': SHARED_SECRET},
        timeout=20.0,
        follow_redirects=True,
    )
    print('Public /health via tunnel:', hr.status_code, hr.text[:300])
except Exception as e:
    print('Public health probe failed (tunnel may still be warming):', e)

print()
print('Giữ runtime Colab + cell này đang chạy. Đóng tab / disconnect = tunnel chết.')

## 5) Checklist desktop

Trong **Video Clone → Cấu hình**:

| Field | Giá trị |
|-------|--------|
| Backend mode | `Remote` |
| Remote URL | `REMOTE_URL` in ở cell 4 |
| Shared secret | `SHARED_SECRET` in ở cell 2 |

Readiness OK khi `GET {REMOTE_URL}/health` + header `X-VC-Secret` trả `{"ok": true}`.

> **Lưu ý MVP:** job ASR/TTS vẫn chạy trên máy local; Colab worker hiện dùng để **readiness / GPU path sắp wire**. `/infer/asr` còn stub.